In [ ]:
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

ASSET_TABLE = "ehm_fleetstore_prd1eun82719_internal.assetmanagement.aircraftengine"
MASTER_TABLE = "ehm_fleetstore_prd1eun82719_internal.`g700-pearl700`.`emucontinuousmaster-aircraftengine-cda`"
SCAN_TABLE = "ehm_fleetstore_prd1eun82719_internal.`g700-pearl700`.`emucontinuousscan-cda`"
ENGINE_TYPE_CODE = "Pearl 700"
CHANNELS = ("AC", "BC")
PARAMETER_PREFIX = "Parameter:EVHMU_EEC_"

# Remove legacy Databricks widgets once; linked ipywidgets in the next cell replace them completely.
for legacy_widget_name in (
    "AircraftID",
    "EngineSerialNumber",
    "Flight_Date",
    "num_years",
    "FlightStartDateTime",
):
    try:
        dbutils.widgets.remove(legacy_widget_name)
    except Exception:
        pass

# Normalize identifiers to strings so UI values and table filters use the same representation.
asset_df = (
    spark.table(ASSET_TABLE)
    .where(F.col("EngineTypeCode") == ENGINE_TYPE_CODE)
    .select(
        F.col("LatestAircraftLatestIdentifier").cast("string").alias("AircraftIdentifier"),
        F.col("LatestAircraftId").alias("AircraftId"),
        F.col("EngineSerialNumber").cast("string").alias("EngineSerialNumber"),
        F.col("EngineId").alias("EngineId"),
        F.col("LatestOperatorName").alias("OperatorName"),
    )
    .where(
        F.col("AircraftIdentifier").isNotNull()
        & F.col("AircraftId").isNotNull()
        & F.col("EngineSerialNumber").isNotNull()
    )
    .dropDuplicates()
)

asset_rows = asset_df.collect()
aircraft_options = sorted({row.AircraftIdentifier for row in asset_rows})
if not aircraft_options:
    raise RuntimeError(f"No aircraft were found for engine type {ENGINE_TYPE_CODE}.")

print(f"Loaded {len(aircraft_options)} aircraft with {ENGINE_TYPE_CODE} engines.")

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output, display as ipy_display

# Build validated aircraft and engine maps once so changing an aircraft only updates widget options.
aircraft_ids_by_identifier = {}
engine_ids_by_aircraft = {}
for row in asset_rows:
    aircraft_ids_by_identifier.setdefault(row.AircraftIdentifier, set()).add(row.AircraftId)
    aircraft_engines = engine_ids_by_aircraft.setdefault(row.AircraftIdentifier, {})
    esn = str(row.EngineSerialNumber)
    if esn in aircraft_engines and aircraft_engines[esn] != row.EngineId:
        raise ValueError(f"Engine serial number {esn} maps to more than one engine ID.")
    aircraft_engines[esn] = row.EngineId

for aircraft_identifier, internal_ids in aircraft_ids_by_identifier.items():
    if len(internal_ids) != 1:
        raise ValueError(
            f"Aircraft {aircraft_identifier} maps to {len(internal_ids)} current internal aircraft IDs."
        )

existing_aircraft_widget = globals().get("aircraft_widget")
existing_engine_widget = globals().get("engine_widget")
existing_start_widget = globals().get("start_date_widget")
existing_end_widget = globals().get("end_date_widget")

previous_aircraft = getattr(existing_aircraft_widget, "value", None)
previous_engine = getattr(existing_engine_widget, "value", None)
default_aircraft = previous_aircraft if previous_aircraft in aircraft_options else aircraft_options[0]
default_engine_options = sorted(engine_ids_by_aircraft[default_aircraft])
default_engine = previous_engine if previous_engine in default_engine_options else default_engine_options[0]

# Preserve an existing UI range; otherwise open on the latest seven-day period containing data.
previous_start_date = getattr(existing_start_widget, "value", None)
previous_end_date = getattr(existing_end_widget, "value", None)
if previous_start_date is None or previous_end_date is None:
    latest_calendar_row = (
        spark.table(MASTER_TABLE)
        .where(
            (F.col("AircraftIdentifier").cast("string") == default_aircraft)
            & (F.col("EngineSerialNumber").cast("string") == default_engine)
        )
        .agg(F.max("CalendarId").alias("LatestCalendarId"))
        .first()
    )
    latest_calendar_id = latest_calendar_row.LatestCalendarId if latest_calendar_row else None
    previous_end_date = (
        datetime.strptime(str(int(latest_calendar_id)), "%Y%m%d").date()
        if latest_calendar_id is not None
        else date.today()
    )
    previous_start_date = previous_end_date - timedelta(days=6)

aircraft_widget = widgets.Dropdown(
    options=aircraft_options,
    value=default_aircraft,
    description="Aircraft:",
    layout=widgets.Layout(width="420px"),
)
engine_widget = widgets.Dropdown(
    options=default_engine_options,
    value=default_engine,
    description="Engine:",
    layout=widgets.Layout(width="420px"),
)
start_date_widget = widgets.DatePicker(
    value=previous_start_date,
    description="Start date:",
    layout=widgets.Layout(width="300px"),
)
end_date_widget = widgets.DatePicker(
    value=previous_end_date,
    description="End date:",
    layout=widgets.Layout(width="300px"),
)
load_flights_btn = widgets.Button(
    description="Load selected flights",
    button_style="primary",
    layout=widgets.Layout(width="220px"),
)
selection_output = widgets.Output()

# Update the engine choices in place; no widget is removed or recreated.
def _on_aircraft_change(change):
    new_aircraft = change["new"]
    new_engine_options = sorted(engine_ids_by_aircraft[new_aircraft])
    current_engine = engine_widget.value
    engine_widget.options = new_engine_options
    engine_widget.value = (
        current_engine if current_engine in new_engine_options else new_engine_options[0]
    )


def _month_partitions_between_dates(start_date, end_date):
    current = date(start_date.year, start_date.month, 1)
    final = date(end_date.year, end_date.month, 1)
    partitions = []
    while current <= final:
        partitions.append((current.year, current.month))
        if current.month == 12:
            current = date(current.year + 1, 1, 1)
        else:
            current = date(current.year, current.month + 1, 1)
    return partitions


# Load every selected-engine flight whose master record starts inside the chosen date range.
def _load_selected_flights(_, raise_errors=False):
    load_flights_btn.disabled = True
    try:
        with selection_output:
            clear_output(wait=True)
            chosen_aircraft = aircraft_widget.value
            chosen_engine = engine_widget.value
            chosen_start_date = start_date_widget.value
            chosen_end_date = end_date_widget.value

            if chosen_start_date is None or chosen_end_date is None:
                raise ValueError("Both start and end dates are required.")
            if chosen_start_date > chosen_end_date:
                raise ValueError("Start date must not be later than end date.")

            internal_ids = aircraft_ids_by_identifier[chosen_aircraft]
            chosen_internal_aircraft_id = next(iter(internal_ids))
            chosen_engine_id = engine_ids_by_aircraft[chosen_aircraft][chosen_engine]
            start_calendar_id = int(chosen_start_date.strftime("%Y%m%d"))
            end_calendar_id = int(chosen_end_date.strftime("%Y%m%d"))

            master_df = (
                spark.table(MASTER_TABLE)
                .where(
                    (F.col("AircraftIdentifier").cast("string") == chosen_aircraft)
                    & (F.col("EngineSerialNumber").cast("string") == chosen_engine)
                    & F.col("CalendarId").between(start_calendar_id, end_calendar_id)
                )
            )
            windows_df = (
                master_df.select("StartDatetime", "EndDatetime")
                .where(F.col("StartDatetime").isNotNull() & F.col("EndDatetime").isNotNull())
                .groupBy("StartDatetime")
                .agg(F.max("EndDatetime").alias("EndDatetime"))
                .where(F.col("EndDatetime") >= F.col("StartDatetime"))
            )
            chosen_flight_rows = windows_df.orderBy(F.col("StartDatetime")).collect()

            chosen_scan_table = spark.table(SCAN_TABLE)
            if chosen_flight_rows:
                range_start = min(row.StartDatetime for row in chosen_flight_rows)
                range_end = max(row.EndDatetime for row in chosen_flight_rows)
                partition_filter = F.lit(False)
                for partition_year, partition_month in _month_partitions_between_dates(
                    range_start.date(),
                    range_end.date(),
                ):
                    partition_filter = partition_filter | (
                        (F.col("year") == partition_year)
                        & (F.col("Month") == partition_month)
                    )

                scan_alias = chosen_scan_table.where(
                    (F.col("AircraftId") == chosen_internal_aircraft_id)
                    & (F.col("AssetIdentifier").cast("string") == chosen_engine)
                    & partition_filter
                ).alias("scan")
                window_alias = F.broadcast(windows_df).alias("window")
                join_condition = (
                    (scan_alias["StartDatetime"] == window_alias["StartDatetime"])
                    & scan_alias["Timestamp"].between(
                        window_alias["StartDatetime"],
                        window_alias["EndDatetime"],
                    )
                )
                chosen_cda = (
                    scan_alias.join(window_alias, join_condition, "inner")
                    .select(*[scan_alias[column] for column in chosen_scan_table.columns])
                )
                total_flight_duration = sum(
                    (row.EndDatetime - row.StartDatetime for row in chosen_flight_rows),
                    timedelta(0),
                )
            else:
                range_start = None
                range_end = None
                total_flight_duration = timedelta(0)
                chosen_cda = chosen_scan_table.limit(0)

            # Commit the new state only after every validation and query-planning step succeeds.
            globals().update(
                {
                    "selected_acid": chosen_aircraft,
                    "selected_esn": chosen_engine,
                    "internal_acid": chosen_internal_aircraft_id,
                    "engineid": chosen_engine_id,
                    "window_start_date": chosen_start_date,
                    "window_end_date": chosen_end_date,
                    "start_calendar_id": start_calendar_id,
                    "end_calendar_id": end_calendar_id,
                    "cda_master_dt": master_df,
                    "flight_windows_df": windows_df,
                    "flight_rows": chosen_flight_rows,
                    "flight_available": bool(chosen_flight_rows),
                    "selected_start_dt": range_start,
                    "selected_end_dt": range_end,
                    "start_dt": range_start,
                    "end_dt": range_end,
                    "flight_duration": total_flight_duration,
                    "scan_table": chosen_scan_table,
                    "cda_cols": chosen_scan_table.columns,
                    "cda": chosen_cda,
                    "selection_revision": globals().get("selection_revision", 0) + 1,
                }
            )

            print(
                f"Aircraft: {chosen_aircraft} | Internal aircraft ID: "
                f"{chosen_internal_aircraft_id} | ESN: {chosen_engine} | Engine ID: {chosen_engine_id}"
            )
            print(f"Date range: {chosen_start_date} through {chosen_end_date}")
            print(
                f"Flights loaded: {len(chosen_flight_rows)} | "
                f"Combined flight duration: {total_flight_duration}"
            )
            if chosen_flight_rows:
                print("Run Cell 3 and the cells below it to refresh data, plots, and correlations.")
            else:
                print("No matching flights were found for this selection.")
    except Exception as exc:
        with selection_output:
            print(f"Unable to load the selected flights: {exc}")
        if raise_errors:
            raise
    finally:
        load_flights_btn.disabled = False


aircraft_widget.observe(_on_aircraft_change, names="value")
load_flights_btn.on_click(_load_selected_flights)

selection_controls = widgets.VBox(
    [
        widgets.HBox([aircraft_widget, engine_widget]),
        widgets.HBox([start_date_widget, end_date_widget, load_flights_btn]),
        selection_output,
    ]
)
ipy_display(selection_controls)

# Load the defaults during a full notebook run; later changes use the button above.
_load_selected_flights(None, raise_errors=True)

In [ ]:
# Define channel-neutral suffixes once, then generate matching AC and BC columns programmatically.
THRUST_REVERSER_SUFFIXES = [
    "IAcThrustReverser_tcmIntlV_data",
    "IAcThrustReverser_trLeftDoorSwLocked_LOWER__data",
    "IAcThrustReverser_trLeftDoorSwLocked_UPPER__data",
    "IAcThrustReverser_trLwrDoorLeftSwLocked_data",
    "IAcThrustReverser_trLwrDoorRightSwLocked_data",
    "IAcThrustReverser_trLwrDoorRightSwLocked_flt",
    "IAcThrustReverser_trRightDoorSwLocked_LOWER__data",
    "IAcThrustReverser_trRightDoorSwLocked_UPPER__data",
    "IAcThrustReverser_trUprDoorLeftSwLocked_data",
    "IAcThrustReverser_trUprDoorLeftSwLocked_flt",
    "IAcThrustReverser_trUprDoorRightSwLocked_data",
    "IAcThrustReverser_trUprDoorRightSwLocked_flt",
    "IOSThrustReverser_mainTestEnabled_data",
    "IOSThrustReverser_trDoorPosVa_data",
    "IOSThrustReverser_trDoorPosVb_data",
    "IOSThrustReverser_trDoorPosVex_data",
    "IOSThrustReverser_trDoorPosVex_extflt",
    "IOSThrustReverser_trDoorPosVex_intflt",
    "IOSThrustReverser_trLoSwRaw_data",
    "IOSThrustReverser_trUpSwRaw_data",
    "IOtherProcThrustReverser_trLoLVTSelected_data",
    "IOtherProcThrustReverser_trUpLVTSelected_data",
    "IThrustReverser_loOwnTRLVTRaw_data",
    "IThrustReverser_loOwnTRLVTRngFlt_data",
    "IThrustReverser_trAnomaly_data",
    "IThrustReverser_trArmed_data",
    "IThrustReverser_trDeploySelected_data",
    "IThrustReverser_trDoorDeplStatus_data",
    "IThrustReverser_trDoorPosLost_data",
    "IThrustReverser_trDoorStowStatus_data",
    "IThrustReverser_trInTransit_data",
    "IThrustReverser_trInadvDeploy_data",
    "IThrustReverser_trIsDeployed_data",
    "IThrustReverser_trIsUnlocked_data",
    "IThrustReverser_trJam_data",
    "IThrustReverser_trLVTPosVCtrl_data",
    "IThrustReverser_trLegalDeployCmd_data",
    "IThrustReverser_trLessDeplPos_data",
    "IThrustReverser_trLoLVTSelected_data",
    "IThrustReverser_trLoLVTSelected_flt",
    "IThrustReverser_trMTESTimeExpired_data",
    "IThrustReverser_trMTESV_data",
    "IThrustReverser_trMoreDeplPos_data",
    "IThrustReverser_trUnavailable_data",
    "IThrustReverser_trUpLVTSelected_data",
    "IThrustReverser_trUpLVTSelected_flt",
    "IThrustReverser_trVPrSwVFlt_data",
    "IThrustReverser_trVPrSwV_data",
    "IThrustReverser_upOwnTRLVTRaw_data",
    "IThrustReverser_upOwnTRLVTRngFlt_data",
]

THRUST_REVERSER_COLUMNS = [
    f"{PARAMETER_PREFIX}{channel}_{suffix}"
    for channel in CHANNELS
    for suffix in THRUST_REVERSER_SUFFIXES
]
CONTEXT_COLUMNS = [
    f"{PARAMETER_PREFIX}AC_IHPShaft_nhV_data",
    f"{PARAMETER_PREFIX}BC_IHPShaft_nhV_data",
    f"{PARAMETER_PREFIX}AC_IAircraftState_altitudeC_data",
    f"{PARAMETER_PREFIX}BC_IAircraftState_altitudeC_data",
]
IDENTIFIER_COLUMNS = ["Timestamp", "StartDatetime", "ParentAssetIdentifier", "AssetIdentifier"]

# Treat identifiers as mandatory while allowing optional signals to be absent from a schema revision.
scan_schema = set(cda_cols)
missing_identifiers = [column for column in IDENTIFIER_COLUMNS if column not in scan_schema]
if missing_identifiers:
    raise ValueError(f"The scan table is missing required columns: {missing_identifiers}")

requested_parameter_columns = THRUST_REVERSER_COLUMNS + CONTEXT_COLUMNS
unavailable_parameter_columns = [
    column for column in requested_parameter_columns if column not in scan_schema
]
PARAMETER_COLUMNS = [column for column in requested_parameter_columns if column in scan_schema]
selected_cols = IDENTIFIER_COLUMNS + PARAMETER_COLUMNS

if unavailable_parameter_columns:
    print(f"Unavailable optional parameters: {len(unavailable_parameter_columns)}")

# Materialize only the chosen engine and columns after Spark has applied every flight filter.
select_para_df = cda.select(*selected_cols).orderBy(F.col("Timestamp"))
MAX_PANDAS_ROWS = 1_000_000
selected_row_count = select_para_df.count()
if selected_row_count > MAX_PANDAS_ROWS:
    raise RuntimeError(
        f"The selected range contains {selected_row_count:,} samples. "
        f"Reduce the date range below {MAX_PANDAS_ROWS:,} samples before converting to Pandas."
    )
_data_df = select_para_df.toPandas()

if not _data_df.empty:
    _data_df["Timestamp"] = pd.to_datetime(_data_df["Timestamp"], errors="coerce")
    _data_df["StartDatetime"] = pd.to_datetime(_data_df["StartDatetime"], errors="coerce")
    _data_df = _data_df.sort_values("Timestamp", kind="stable").reset_index(drop=True)

print(f"Loaded samples: {len(_data_df):,}")
print(f"Loaded parameter columns: {len(PARAMETER_COLUMNS)}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display as ipy_display

# Keep short names in the UI while retaining an unambiguous mapping to the Spark columns.
def _short_parameter_name(column):
    return column[len(PARAMETER_PREFIX):] if column.startswith(PARAMETER_PREFIX) else column


combined_channel_columns = [_short_parameter_name(column) for column in PARAMETER_COLUMNS]
FEATURE_COLUMNS = [
    _short_parameter_name(column)
    for column in CONTEXT_COLUMNS
    if column in PARAMETER_COLUMNS
]

input_search = widgets.Combobox(
    placeholder="Search and add a parameter",
    options=[option for option in combined_channel_columns if option not in FEATURE_COLUMNS],
    description="Add input:",
    ensure_option=True,
    layout=widgets.Layout(width="650px"),
)
input_display = widgets.SelectMultiple(
    options=tuple(FEATURE_COLUMNS),
    description="Plot inputs:",
    layout=widgets.Layout(width="650px", height="280px"),
)
remove_input_btn = widgets.Button(
    description="Remove highlighted",
    button_style="danger",
    icon="trash",
)
apply_input_btn = widgets.Button(
    description="Apply inputs",
    button_style="info",
)
selected_output = widgets.Output()

# Mutate this list in place so plotting callbacks always see the latest applied inputs.
selected_parameters = list(FEATURE_COLUMNS)


def _refresh_search_options():
    active = set(input_display.options)
    input_search.options = [
        option for option in combined_channel_columns if option not in active
    ]


def _sync_selected_parameters():
    selected_parameters[:] = list(input_display.options)


def add_to_input(change):
    value = change["new"]
    if value and value in combined_channel_columns and value not in input_display.options:
        input_display.options = tuple(input_display.options) + (value,)
        _sync_selected_parameters()
        input_search.value = ""
        _refresh_search_options()


def remove_input(_):
    highlighted = set(input_display.value)
    input_display.options = tuple(
        option for option in input_display.options if option not in highlighted
    )
    _sync_selected_parameters()
    _refresh_search_options()


def apply_inputs(_):
    _sync_selected_parameters()
    with selected_output:
        selected_output.clear_output()
        print(f"Applied {len(selected_parameters)} plot inputs.")


input_search.observe(add_to_input, names="value")
remove_input_btn.on_click(remove_input)
apply_input_btn.on_click(apply_inputs)

input_column = widgets.VBox(
    [input_search, input_display, remove_input_btn, apply_input_btn, selected_output]
)
ipy_display(input_column)

In [ ]:
from collections import OrderedDict

import ipywidgets as widgets
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from IPython.display import clear_output, display as ipy_display
from ipywidgets import SelectionRangeSlider

# Reapply the ESN filter defensively in case _data_df was supplied by an earlier notebook run.
data_df = _data_df.copy()
if "AssetIdentifier" in data_df.columns:
    data_df = data_df.loc[
        data_df["AssetIdentifier"].astype("string") == str(selected_esn)
    ].copy()
if "Timestamp" in data_df.columns:
    data_df["Timestamp"] = pd.to_datetime(data_df["Timestamp"], errors="coerce")
    data_df = data_df.dropna(subset=["Timestamp"]).sort_values("Timestamp", kind="stable")

MAX_SLIDER_POINTS = 1000
MAX_SIGNAL_GROUPS = 16
CHANNEL_COLORS = {"AC": "#1f77b4", "BC": "#ff7f0e"}


def _full_parameter_name(short_name):
    return short_name if short_name.startswith(PARAMETER_PREFIX) else PARAMETER_PREFIX + short_name


def _channel_and_signal(short_name):
    for channel in CHANNELS:
        prefix = f"{channel}_"
        if short_name.startswith(prefix):
            return channel, short_name[len(prefix):]
    return "", short_name


def _numeric_series(series):
    non_null = series.dropna()
    if non_null.empty:
        return pd.Series(np.nan, index=series.index, dtype="float64"), False
    is_boolean = non_null.map(lambda value: isinstance(value, (bool, np.bool_))).all()
    if is_boolean:
        return series.map({True: 1.0, False: 0.0}), True
    numeric = pd.to_numeric(series, errors="coerce")
    values = numeric.dropna().to_numpy(dtype=float)
    is_discrete = bool(
        len(values)
        and len(np.unique(values)) <= 8
        and np.allclose(values, np.round(values))
    )
    return numeric, is_discrete


def _match_timestamp_timezone(value, reference):
    timestamp = pd.Timestamp(value)
    if reference.tzinfo is None and timestamp.tzinfo is not None:
        return timestamp.tz_localize(None)
    if reference.tzinfo is not None and timestamp.tzinfo is None:
        return timestamp.tz_localize(reference.tzinfo)
    return timestamp


def _plot_parameters(start_time, end_time):
    chosen = list(selected_parameters)
    if not chosen:
        print("No plot inputs are applied. Add at least one parameter in the previous cell.")
        return

    grouped = OrderedDict()
    for short_name in chosen:
        channel, signal = _channel_and_signal(short_name)
        grouped.setdefault(signal, []).append((channel, short_name))

    if len(grouped) > MAX_SIGNAL_GROUPS:
        print(
            f"The current selection creates {len(grouped)} signal groups. "
            f"Reduce it to {MAX_SIGNAL_GROUPS} or fewer for a readable plot."
        )
        return

    visible = data_df.loc[
        data_df["Timestamp"].between(start_time, end_time, inclusive="both")
    ].copy()
    if visible.empty:
        print("No samples fall inside the requested time range.")
        return

    figure_height = max(4.5, 3.0 * len(grouped))
    figure, axes = plt.subplots(
        len(grouped),
        1,
        figsize=(22, figure_height),
        sharex=True,
        squeeze=False,
    )
    skipped = []

    # Put AC and BC versions of the same signal on one axis for direct channel comparison.
    for axis, (signal, members) in zip(axes[:, 0], grouped.items()):
        plotted = False
        for channel, short_name in members:
            full_name = _full_parameter_name(short_name)
            if full_name not in visible.columns:
                skipped.append(short_name)
                continue

            parameter_frame = visible[["Timestamp", "StartDatetime", full_name]].copy()
            parameter_frame["_value"], is_discrete = _numeric_series(parameter_frame[full_name])
            parameter_frame = parameter_frame.dropna(subset=["_value"])
            if parameter_frame.empty:
                skipped.append(short_name)
                continue

            # Plot each flight separately so lines do not bridge overnight or inter-flight gaps.
            for segment_index, (_, flight_segment) in enumerate(
                parameter_frame.groupby("StartDatetime", dropna=False, sort=True)
            ):
                axis.plot(
                    flight_segment["Timestamp"],
                    flight_segment["_value"],
                    color=CHANNEL_COLORS.get(channel, "#333333"),
                    linewidth=1.2,
                    drawstyle="steps-post" if is_discrete else "default",
                    label=(channel or short_name) if segment_index == 0 else "_nolegend_",
                )
            plotted = True

        axis.set_ylabel(signal, fontsize=9)
        axis.grid(True, alpha=0.25)
        if plotted:
            axis.legend(loc="upper right", ncol=max(1, len(members)))
        else:
            axis.text(0.5, 0.5, "No numeric samples", ha="center", va="center", transform=axis.transAxes)

    axes[-1, 0].set_xlabel("Timestamp")
    date_locator = mdates.AutoDateLocator(minticks=4, maxticks=12)
    axes[-1, 0].xaxis.set_major_locator(date_locator)
    axes[-1, 0].xaxis.set_major_formatter(mdates.ConciseDateFormatter(date_locator))
    figure.suptitle(
        f"Aircraft {selected_acid} | Engine {selected_esn} | "
        f"{start_time} through {end_time}",
        fontsize=14,
    )
    figure.tight_layout(rect=(0, 0, 1, 0.98))
    plt.show()
    plt.close(figure)

    if skipped:
        print("Skipped parameters without numeric samples: " + ", ".join(sorted(set(skipped))))


if data_df.empty:
    print("No samples are available for the selected flights and engine.")
else:
    all_timestamps = [pd.Timestamp(value) for value in sorted(data_df["Timestamp"].unique())]
    if len(all_timestamps) > MAX_SLIDER_POINTS:
        indices = np.linspace(
            0,
            len(all_timestamps) - 1,
            num=MAX_SLIDER_POINTS,
            dtype=int,
        )
        slider_timestamps = [all_timestamps[index] for index in np.unique(indices)]
    else:
        slider_timestamps = all_timestamps

    def _timestamp_label(timestamp):
        return timestamp.strftime("%Y-%m-%d %H:%M:%S.%f").rstrip("0").rstrip(".")


    time_slider = SelectionRangeSlider(
        options=[(_timestamp_label(timestamp), timestamp) for timestamp in slider_timestamps],
        index=(0, len(slider_timestamps) - 1),
        description="Time range:",
        orientation="horizontal",
        layout=widgets.Layout(width="1200px"),
        continuous_update=False,
    )
    txt_start = widgets.Text(
        value=_timestamp_label(all_timestamps[0]),
        description="Start:",
        layout=widgets.Layout(width="500px"),
    )
    txt_end = widgets.Text(
        value=_timestamp_label(all_timestamps[-1]),
        description="End:",
        layout=widgets.Layout(width="500px"),
    )
    plot_btn = widgets.Button(
        description="Plot",
        button_style="success",
        icon="line-chart",
        layout=widgets.Layout(width="150px"),
    )
    plot_output = widgets.Output()

    def _render_range(start_time, end_time):
        with plot_output:
            clear_output(wait=True)
            if start_time > end_time:
                start_time, end_time = end_time, start_time
            _plot_parameters(start_time, end_time)


    def _on_plot_click(_):
        reference = all_timestamps[0]
        try:
            start_time = _match_timestamp_timezone(txt_start.value, reference)
            end_time = _match_timestamp_timezone(txt_end.value, reference)
        except Exception:
            with plot_output:
                clear_output(wait=True)
                print("Start and end must be valid date-time values.")
            return
        _render_range(start_time, end_time)


    def _on_slider_change(change):
        start_time, end_time = map(pd.Timestamp, change["new"])
        txt_start.value = _timestamp_label(start_time)
        txt_end.value = _timestamp_label(end_time)
        _render_range(start_time, end_time)


    plot_btn.on_click(_on_plot_click)
    time_slider.observe(_on_slider_change, names="value")

    controls = widgets.VBox([txt_start, txt_end, time_slider, plot_btn])
    ipy_display(widgets.VBox([controls, plot_output]))
    _render_range(all_timestamps[0], all_timestamps[-1])

In [ ]:
import math

import ipywidgets as widgets
from IPython.display import clear_output, display as ipy_display
from pyspark.sql.types import BooleanType, NumericType

CORRELATION_BATCH_SIZE = 100
CORRELATION_MAX_INPUTS = 8
CORRELATION_RANDOM_SEED = 1701
CORRELATION_EXCLUDED_FRAGMENTS = (
    "timestamp",
    "datetime",
    "airlineid",
    "aircraftid",
    "engineid",
    "operatorid",
    "serialnumber",
    "assetidentifier",
    "calendarid",
)

# Quote arbitrary Spark column names so punctuation cannot be interpreted as nested-field syntax.
def _quoted_spark_column(name):
    return F.col(f"`{name.replace('`', '``')}`")


def _eligible_correlation_parameter(field):
    normalized = field.name.lower().replace("_", "")
    is_parameter = field.name.startswith("Parameter:")
    is_numeric = isinstance(field.dataType, (NumericType, BooleanType))
    is_metadata = any(fragment in normalized for fragment in CORRELATION_EXCLUDED_FRAGMENTS)
    return is_parameter and is_numeric and not is_metadata


# Correlation is a screening measure, not evidence that one engine signal causes another.
correlation_candidate_columns = [
    field.name for field in cda.schema.fields if _eligible_correlation_parameter(field)
]
correlation_candidate_set = set(correlation_candidate_columns)

top_results_widget = widgets.BoundedIntText(
    value=50,
    min=5,
    max=200,
    step=5,
    description="Top results:",
    layout=widgets.Layout(width="260px"),
)
row_limit_widget = widgets.BoundedIntText(
    value=50000,
    min=1000,
    max=500000,
    step=10000,
    description="Max rows:",
    layout=widgets.Layout(width="260px"),
)
minimum_pairs_widget = widgets.BoundedIntText(
    value=30,
    min=3,
    max=10000,
    step=10,
    description="Minimum pairs:",
    layout=widgets.Layout(width="260px"),
)
run_correlation_btn = widgets.Button(
    description="Run correlation analysis",
    button_style="primary",
    layout=widgets.Layout(width="240px"),
)
correlation_output = widgets.Output()


def _format_ranked_correlations(records, top_count):
    positive = sorted(
        (record for record in records if record[1] > 0),
        key=lambda record: (-record[1], record[0]),
    )[:top_count]
    negative = sorted(
        (record for record in records if record[1] < 0),
        key=lambda record: (record[1], record[0]),
    )[:top_count]

    positive_text = [
        f"{_short_parameter_name(parameter)} - {value:+.6f}"
        for parameter, value, _ in positive
    ]
    negative_text = [
        f"{_short_parameter_name(parameter)} - {value:+.6f}"
        for parameter, value, _ in negative
    ]
    output_length = max(len(positive_text), len(negative_text), 1)
    positive_text.extend([""] * (output_length - len(positive_text)))
    negative_text.extend([""] * (output_length - len(negative_text)))
    return pd.DataFrame(
        {
            "Positive correlations": positive_text,
            "Negative correlations": negative_text,
        }
    )


def _run_correlation_analysis(_):
    run_correlation_btn.disabled = True
    try:
        with correlation_output:
            clear_output(wait=True)

            input_columns = []
            for short_name in selected_parameters:
                full_name = _full_parameter_name(short_name)
                if full_name in correlation_candidate_set and full_name not in input_columns:
                    input_columns.append(full_name)

            if not input_columns:
                print("None of the applied input features are eligible numeric or Boolean parameters.")
                return
            if len(input_columns) > CORRELATION_MAX_INPUTS:
                print(
                    f"Correlation is limited to {CORRELATION_MAX_INPUTS} input features per run. "
                    "Reduce the applied inputs in the parameter-selection cell."
                )
                return

            source_row_count = cda.count()
            if source_row_count == 0:
                print("No flight samples are available for correlation analysis.")
                return

            row_limit = int(row_limit_widget.value)
            sample_fraction = min(1.0, row_limit / source_row_count)
            minimum_pairs = int(minimum_pairs_widget.value)
            top_count = int(top_results_widget.value)
            correlations_by_input = {column: [] for column in input_columns}

            print(
                f"Inputs: {len(input_columns)} | Candidate parameters: "
                f"{len(correlation_candidate_columns):,} | Flight rows: {source_row_count:,}"
            )
            if sample_fraction < 1.0:
                print(
                    f"Using a reproducible random sample of approximately {row_limit:,} rows "
                    "to bound runtime and memory use."
                )

            for batch_start in range(0, len(correlation_candidate_columns), CORRELATION_BATCH_SIZE):
                batch_columns = correlation_candidate_columns[
                    batch_start:batch_start + CORRELATION_BATCH_SIZE
                ]
                required_columns = list(dict.fromkeys(input_columns + batch_columns))
                batch_df = cda.select(
                    *[_quoted_spark_column(column).alias(column) for column in required_columns]
                )
                if sample_fraction < 1.0:
                    batch_df = batch_df.sample(
                        withReplacement=False,
                        fraction=sample_fraction,
                        seed=CORRELATION_RANDOM_SEED,
                    ).limit(row_limit)

                aggregate_expressions = []
                result_keys = []
                for input_index, input_column in enumerate(input_columns):
                    left_raw = _quoted_spark_column(input_column).cast("double")
                    left = F.when(left_raw.isNull() | F.isnan(left_raw), None).otherwise(left_raw)
                    for candidate_index, candidate_column in enumerate(batch_columns):
                        if candidate_column == input_column:
                            continue
                        right_raw = _quoted_spark_column(candidate_column).cast("double")
                        right = F.when(right_raw.isNull() | F.isnan(right_raw), None).otherwise(right_raw)
                        correlation_alias = f"correlation_{input_index}_{candidate_index}"
                        overlap_alias = f"overlap_{input_index}_{candidate_index}"
                        aggregate_expressions.extend(
                            [
                                F.corr(left, right).alias(correlation_alias),
                                F.count(F.when(left.isNotNull() & right.isNotNull(), 1)).alias(overlap_alias),
                            ]
                        )
                        result_keys.append(
                            (
                                input_column,
                                candidate_column,
                                correlation_alias,
                                overlap_alias,
                            )
                        )

                if aggregate_expressions:
                    aggregate_row = batch_df.agg(*aggregate_expressions).first().asDict()
                    for input_column, candidate_column, correlation_alias, overlap_alias in result_keys:
                        correlation_value = aggregate_row.get(correlation_alias)
                        overlap_count = int(aggregate_row.get(overlap_alias) or 0)
                        if (
                            correlation_value is not None
                            and math.isfinite(float(correlation_value))
                            and overlap_count >= minimum_pairs
                        ):
                            correlations_by_input[input_column].append(
                                (candidate_column, float(correlation_value), overlap_count)
                            )

                completed = min(
                    batch_start + CORRELATION_BATCH_SIZE,
                    len(correlation_candidate_columns),
                )
                print(f"Processed {completed:,} of {len(correlation_candidate_columns):,} candidates.")

            for input_column in input_columns:
                print()
                print(f"Input feature: {_short_parameter_name(input_column)}")
                print(
                    f"Pearson correlations with at least {minimum_pairs:,} paired observations. "
                    "Self-correlation and exact zero correlations are omitted."
                )
                ranked_df = _format_ranked_correlations(
                    correlations_by_input[input_column],
                    top_count,
                )
                ipy_display(ranked_df)
    finally:
        run_correlation_btn.disabled = False


run_correlation_btn.on_click(_run_correlation_analysis)
correlation_controls = widgets.HBox(
    [top_results_widget, row_limit_widget, minimum_pairs_widget, run_correlation_btn]
)
ipy_display(widgets.VBox([correlation_controls, correlation_output]))

In [ ]:
# Verify that every requested thrust-reverser suffix has matching AC and BC schema coverage.
schema_columns = set(cda_cols)
selected_parameter_set = set(selected_parameters)
channel_pair_records = []

for suffix in THRUST_REVERSER_SUFFIXES:
    ac_short = f"AC_{suffix}"
    bc_short = f"BC_{suffix}"
    ac_full = PARAMETER_PREFIX + ac_short
    bc_full = PARAMETER_PREFIX + bc_short
    channel_pair_records.append(
        {
            "Signal": suffix,
            "AC available": ac_full in schema_columns,
            "BC available": bc_full in schema_columns,
            "AC selected": ac_short in selected_parameter_set,
            "BC selected": bc_short in selected_parameter_set,
        }
    )

channel_pair_df = pd.DataFrame(channel_pair_records)
unpaired_schema_rows = channel_pair_df.loc[
    channel_pair_df["AC available"] != channel_pair_df["BC available"]
]

print(f"Thrust-reverser signal pairs: {len(channel_pair_df)}")
print(f"Schema pairs with unequal AC/BC availability: {len(unpaired_schema_rows)}")
ipy_display(channel_pair_df)

In [ ]:
# Keep the detailed flight inventory read-only and tied to the active widget selections.
diagnostic_master_columns = [
    "AircraftIdentifier",
    "EngineSerialNumber",
    "AircraftId",
    "EngineId",
    "StartDatetime",
    "EndDatetime",
    "Duration",
    "EnginePosition",
    "OperatorId",
    "OperatorCode",
    "LastGeneratedDatetime",
    "FirstGeneratedDatetime",
    "Changed",
    "Created",
    "Migrated",
    "CalendarId",
]

diagnostic_master_with_duration = cda_master_dt.withColumn(
    "Duration",
    F.col("EndDatetime") - F.col("StartDatetime"),
)
diagnostic_available_columns = [
    column
    for column in diagnostic_master_columns
    if column in diagnostic_master_with_duration.columns
]
diagnostic_master_df = (
    diagnostic_master_with_duration.select(*diagnostic_available_columns)
    .orderBy(F.col("StartDatetime").desc(), F.col("EndDatetime").desc())
)

print(
    f"Flight inventory for aircraft {selected_acid}, engine {selected_esn}, "
    f"from {window_start_date} through {window_end_date}"
)
display(diagnostic_master_df)

In [ ]:
# Show every normalized flight window included in the active multi-day selection.
if flight_available:
    diagnostic_selected_master_df = (
        flight_windows_df.withColumn(
            "Duration",
            F.col("EndDatetime") - F.col("StartDatetime"),
        )
        .orderBy(F.col("StartDatetime"))
    )
    print(
        f"Selected flight windows: {len(flight_rows)} from "
        f"{window_start_date} through {window_end_date}"
    )
else:
    diagnostic_selected_master_df = flight_windows_df.limit(0)
    print("No selected flight windows are available.")

display(diagnostic_selected_master_df)

In [ ]:
# Quantify AC/BC population and exact disagreement without assuming an engineering tolerance.
channel_coverage_records = []
for suffix in THRUST_REVERSER_SUFFIXES:
    ac_column = f"{PARAMETER_PREFIX}AC_{suffix}"
    bc_column = f"{PARAMETER_PREFIX}BC_{suffix}"
    if ac_column not in _data_df.columns or bc_column not in _data_df.columns:
        continue

    ac_values = _data_df[ac_column]
    bc_values = _data_df[bc_column]
    both_present = ac_values.notna() & bc_values.notna()
    exact_disagreement = (
        ac_values.loc[both_present].ne(bc_values.loc[both_present]).fillna(False)
    )
    channel_coverage_records.append(
        {
            "Signal": suffix,
            "AC non-null": int(ac_values.notna().sum()),
            "BC non-null": int(bc_values.notna().sum()),
            "Both non-null": int(both_present.sum()),
            "Exact disagreements": int(exact_disagreement.sum()),
        }
    )

channel_coverage_df = pd.DataFrame(channel_coverage_records)
if channel_coverage_df.empty:
    print("No paired AC/BC samples are available for coverage analysis.")
else:
    print(f"Channel coverage calculated from {len(_data_df):,} flight samples.")
ipy_display(channel_coverage_df)

In [ ]:
# Preview the active flight deterministically without overwriting notebook-wide identifiers.
preview_identifier_columns = [
    column
    for column in IDENTIFIER_COLUMNS
    if column in select_para_df.columns
]
preview_parameter_columns = [
    _full_parameter_name(short_name)
    for short_name in selected_parameters
    if _full_parameter_name(short_name) in select_para_df.columns
]
preview_columns = preview_identifier_columns + [
    column
    for column in preview_parameter_columns
    if column not in preview_identifier_columns
]

diagnostic_preview_df = (
    select_para_df.select(*preview_columns)
    .orderBy(F.col("Timestamp"))
    .limit(50)
)
print(
    f"First 50 ordered samples for aircraft {selected_acid}, engine {selected_esn}, "
    f"{len(flight_rows)} flights from {window_start_date} through {window_end_date}"
)
display(diagnostic_preview_df)